# Load PyTorch Chunk Files (`.pt`)

This notebook loads and inspects mixed human/AI datasets saved by `main.py`.

Configuration is read from `.env`:
- `MINIPILE_OUTPUT_DIR` — where `.pt` chunks are stored
- `GENERATION_MODE` — `append` or `sandwitch`

Each `.pt` file contains:
- `texts` — full mixed text
- `labels` — word-level labels (`0` = human, `1` = AI)
- `models` — OpenRouter model used per sample
- `indices` — original MiniPile index

| Mode | Structure | Label pattern |
|------|-----------|---------------|
| `append` | human prefix + AI continuation | `000...111...` |
| `sandwitch` | human start + AI bridge + human end | `000...111...000...` |

In [55]:
import os
from collections import Counter
from pathlib import Path
import random

import pyarrow.parquet as pq
from dotenv import load_dotenv

from load_pt import (
    MixedTextChunkDataset,
    list_chunks,
    load_chunk,
    load_split,
    make_dataloader,
    summarize_chunk,
)

PROJECT_DIR = Path.cwd()
load_dotenv(PROJECT_DIR / ".env", override=True)


def env_value(key, default=""):
    value = os.environ.get(key, default)
    return value.split("#", 1)[0].strip()


GENERATION_MODE = env_value("GENERATION_MODE", "sandwitch")
output_dir = env_value("MINIPILE_OUTPUT_DIR", "output")
output_path = Path(output_dir)

if output_path.name in ("append", "sandwitch"):
    OUTPUT_DIR = output_path
elif output_path.name == "output" or output_dir == "output":
    OUTPUT_DIR = output_path / GENERATION_MODE
else:
    OUTPUT_DIR = output_path

DATA_DIR = Path(env_value("MINIPILE_DATA_DIR", "dataset/minipile/data"))
SPLIT = env_value("MINIPILE_SPLITS", "train").split(",")[0]
FROM_IDX = int(env_value("MINIPILE_FROM", "0"))
TO_IDX = int(env_value("MINIPILE_TO")) if env_value("MINIPILE_TO") else None

print(f"Output dir:       {OUTPUT_DIR}")
print(f"Generation mode:  {GENERATION_MODE}")
print(f"Data dir:         {DATA_DIR}")
print(f"Split:            {SPLIT}")
print(f"Index range:      [{FROM_IDX}, {TO_IDX})")


def get_original_text(data_dir: Path, split: str, index: int) -> str:
    files = sorted(data_dir.glob(f"{split}-*.parquet"))
    seen = 0
    for file_path in files:
        table = pq.read_table(file_path, columns=["text"])
        n = table.num_rows
        if seen + n > index:
            return table.column("text")[index - seen].as_py()
        seen += n
    raise IndexError(f"Index {index} out of range for split '{split}'")


def detect_mode(labels) -> str:
    labels = [int(x) for x in labels]
    if 1 not in labels:
        return "unknown"
    last_one = len(labels) - 1 - labels[::-1].index(1)
    trailing_human = any(label == 0 for label in labels[last_one + 1 :])
    return "sandwitch" if trailing_human else "append"


def split_by_labels(text: str, labels, mode=None):
    words = text.split()
    labels = [int(x) for x in labels]
    mode = mode or GENERATION_MODE
    segments = {"human_start": [], "ai": [], "human_end": []}

    for word, label in zip(words, labels):
        if label == 0:
            if mode == "append" and segments["ai"]:
                segments["human_end"].append(word)
            else:
                segments["human_start"].append(word)
        else:
            segments["ai"].append(word)

    if mode == "append":
        segments["human_end"] = []
    return mode, segments


def compare_with_original(item, data_dir: Path, split: str):
    original = get_original_text(data_dir, split, item["index"])
    original_words = original.split()
    mode, segments = split_by_labels(item["text"], item["labels"])
    detected = detect_mode(item["labels"])

    print(f"Index: {item['index']}")
    print(f"OpenRouter model: {item['model']}")
    print(f"Expected mode (.env): {GENERATION_MODE}")
    print(f"Detected mode (labels): {detected}")
    if detected != GENERATION_MODE and detected != "unknown":
        print("Warning: detected mode differs from GENERATION_MODE in .env")
    print(
        f"Label counts: human_start={len(segments['human_start'])}, "
        f"ai={len(segments['ai'])}, human_end={len(segments['human_end'])}"
    )

    print("\n--- Original")
    print(original)

    if mode == "append":
        n = len(segments["human_start"])
        print(f"\n--- Original prefix [{n} words] ---")
        print(" ".join(original_words[:n]))
        print("\n--- Recovered human prefix ---")
        print(" ".join(segments["human_start"]))
        print("\n--- AI continuation ---")
        print(" ".join(segments["ai"]))
    else:
        n_start = len(segments["human_start"])
        n_end = len(segments["human_end"])
        print("\n--- AI bridge ---")
        print(" ".join(segments["ai"]))
        if n_start + n_end < len(original_words):
            skipped = original_words[n_start : len(original_words) - n_end]
            print(f"\n--- Skipped original middle ({len(skipped)} words) ---")
            print(" ".join(skipped[:40]) + ("..." if len(skipped) > 40 else ""))

    return mode, segments


def show_labeled_segments(text: str, labels):
    mode, segments = split_by_labels(text, labels)
    print(f"Mode (.env): {mode}")
    print(
        f"human_start ({len(segments['human_start'])}): "
        f"{' '.join(segments['human_start'][:15])}{'...' if len(segments['human_start']) > 15 else ''}"
    )
    print(
        f"ai          ({len(segments['ai'])}): "
        f"{' '.join(segments['ai'][:15])}{'...' if len(segments['ai']) > 15 else ''}"
    )
    if segments["human_end"]:
        print(
            f"human_end   ({len(segments['human_end'])}): "
            f"{' '.join(segments['human_end'][:15])}{'...' if len(segments['human_end']) > 15 else ''}"
        )

Output dir:       output/sandwitch_v2
Generation mode:  sandwitch_v2
Data dir:         dataset/minipile/data
Split:            train
Index range:      [0, 1000)


## 1. List available chunk files

In [56]:
chunks = list_chunks(OUTPUT_DIR, SPLIT)
print(f"Found {len(chunks)} chunk(s) for split '{SPLIT}':")
for path in chunks:
    print(f"  {path}")

Found 2 chunk(s) for split 'train':
  output/sandwitch_v2/train/train_0_0.pt
  output/sandwitch_v2/train/train_0_99.pt


## 2. Summarize chunk + pick a sample

In [57]:
PT_PATH = chunks[1] if chunks else OUTPUT_DIR / SPLIT / "train_0_999.pt"

info = summarize_chunk(PT_PATH)
print(f"Path:       {info['path']}")
print(f"Samples:    {info['samples']}")
print(f"Indices:    {info['index_range'][0]}–{info['index_range'][1]}")
print(f"Model set:  {', '.join(info['models'])}")

Path:       output/sandwitch_v2/train/train_0_99.pt
Samples:    100
Indices:    0–99
Model set:  cohere/command-r7b-12-2024+google/gemma-3-4b-it, cohere/command-r7b-12-2024+meta-llama/llama-3.1-8b-instruct, cohere/command-r7b-12-2024+mistralai/mistral-nemo, cohere/command-r7b-12-2024+mistralai/mistral-small-24b-instruct-2501, cohere/command-r7b-12-2024+sao10k/l3-lunaris-8b, deepseek/deepseek-v4-flash+cohere/command-r7b-12-2024, deepseek/deepseek-v4-flash+google/gemma-3-4b-it, deepseek/deepseek-v4-flash+mistralai/mistral-nemo, deepseek/deepseek-v4-flash+mistralai/mistral-small-24b-instruct-2501, deepseek/deepseek-v4-flash+sao10k/l3-lunaris-8b, google/gemma-3-4b-it+cohere/command-r7b-12-2024, google/gemma-3-4b-it+google/gemma-3-4b-it, google/gemma-3-4b-it+meta-llama/llama-3.1-8b-instruct, google/gemma-3-4b-it+mistralai/mistral-nemo, google/gemma-3-4b-it+mistralai/mistral-small-24b-instruct-2501, google/gemma-3-4b-it+sao10k/l3-lunaris-8b, meta-llama/llama-3.1-8b-instruct+cohere/command-r7

## 3. Load one chunk directly

In [58]:
data = load_chunk(PT_PATH)
print("Keys:", list(data.keys()))
print("Sample count:", len(data["texts"]))

Keys: ['texts', 'labels', 'models', 'indices']
Sample count: 100


In [59]:
len(chunks)

2

## 4. Compare with original text (append or sandwitch)

In [ ]:
TARGET_IN_CHUNK = random.randint(0, 100)  # change to inspect another sample
item = MixedTextChunkDataset(PT_PATH)[TARGET_IN_CHUNK]
mode, segments = compare_with_original(item, DATA_DIR, SPLIT)

print("\n--- Full mixed text ---")
print(item["text"])
print("\n--- Labels ---")
print(item["labels"].tolist())

Index: 0
OpenRouter model: meta-llama/llama-3.1-8b-instruct+google/gemma-3-4b-it
Expected mode (.env): sandwitch_v2
Detected mode (labels): sandwitch
Label counts: human_start=83, ai=62, human_end=0

--- Original
HTC's Vive Pro headset is available to pre-order for $799

We've seen plenty of Beats-focused KIRFs in our time, some better than others. Few, however, play quite so directly on the name as OrigAudio's Beets. For $25, adopters get a set of headphones that bear little direct resemblance to Dr. Dre's audio gear of choice, but are no doubt bound to impress friends -- at least, up until they see a root vegetable logo instead of a lower-case B. Thankfully, there's more to it than just amusing and confusing peers. Every purchase will lead to a donation of canned beets (what else?) to the Second Harvest Food Bank of Orange County. For us, that's reason enough to hope that Beats doesn't put the kibosh on OrigAudio's effort. Besides, we could use some accompaniment for our BeetBox.

--

## 5. Scan mode distribution in a chunk

In [12]:
ds = MixedTextChunkDataset(PT_PATH)
mode_counts = Counter(detect_mode(ds[i]["labels"]) for i in range(len(ds)))

print(f"Chunk: {PT_PATH}")
print(f"Total samples: {len(ds)}")
for mode, count in sorted(mode_counts.items()):
    print(f"  {mode}: {count} ({100 * count / len(ds):.1f}%)")

seen = set()
for i in range(len(ds)):
    mode = detect_mode(ds[i]["labels"])
    if mode in seen or mode == "unknown":
        continue
    seen.add(mode)
    print(f"\nExample {mode} sample (index {ds[i]['index']}):")
    show_labeled_segments(ds[i]["text"], ds[i]["labels"])

Chunk: output/append/train/train_9000_9999.pt
Total samples: 1000
  append: 1000 (100.0%)

Example append sample (index 9000):
Mode (.env): append
human_start (96): The U.S. envoy for efforts to end the Ukrainian conflict, Kurt Volker, is set to...
ai          (60): Since the conflict began in 2014, thousands of people have been killed and over a...


## 6. Compare append vs sandwitch across chunks

In [13]:
for path in chunks[:3] + chunks[-1:]:
    chunk_ds = MixedTextChunkDataset(path)
    counts = Counter(detect_mode(chunk_ds[i]["labels"]) for i in range(len(chunk_ds)))
    print(f"\n{path.name}: append={counts.get('append', 0)}, sandwitch={counts.get('sandwitch', 0)}")


train_0_999.pt: append=1000, sandwitch=0

train_1000_1999.pt: append=1000, sandwitch=0

train_2000_2999.pt: append=1000, sandwitch=0

train_9000_9999.pt: append=1000, sandwitch=0


## 7. Load a split with index range filter

In [14]:
subset = load_split(OUTPUT_DIR, SPLIT, from_idx=FROM_IDX, to_idx=TO_IDX)
print(f"Subset size: {len(subset)}")
print(f"First index: {subset[0]['index']}")
print(f"Last index:  {subset[-1]['index']}")

Subset size: 1000
First index: 0
Last index:  999


## 8. DataLoader with padded labels

In [15]:
loader = make_dataloader(
    OUTPUT_DIR,
    SPLIT,
    batch_size=4,
    shuffle=False,
    from_idx=FROM_IDX,
    to_idx=TO_IDX,
)

batch = next(iter(loader))
print("Batch keys:", batch.keys())
print("Texts:", len(batch["text"]))
print("Labels shape:", batch["labels"].shape)
print("Indices:", batch["index"].tolist())
print("Models:", batch["model"])

Batch keys: dict_keys(['text', 'labels', 'model', 'index'])
Texts: 4
Labels shape: torch.Size([4, 144])
Indices: [0, 1, 2, 3]
Models: ['openrouter/owl-alpha', 'openrouter/owl-alpha', 'openrouter/owl-alpha', 'openai/gpt-oss-20b:free']


## 9. Inspect labeled segments

In [16]:
sample_paths = [chunks[0], chunks[-1]] if len(chunks) >= 2 else chunks
for path in sample_paths:
    if path is None or not path.exists():
        continue
    chunk_ds = MixedTextChunkDataset(path)
    sample = chunk_ds[0]
    print(f"\n=== {path.name} | index {sample['index']} | mode {GENERATION_MODE} ===")
    show_labeled_segments(sample["text"], sample["labels"])


=== train_0_999.pt | index 0 | mode append ===
Mode (.env): append
human_start (91): HTC's Vive Pro headset is available to pre-order for $799 We've seen plenty of Beats-focused...
ai          (48): pair delivers surprisingly rich audio quality that belies its playful appearance. The sound profile leans...

=== train_9000_9999.pt | index 9000 | mode append ===
Mode (.env): append
human_start (96): The U.S. envoy for efforts to end the Ukrainian conflict, Kurt Volker, is set to...
ai          (60): Since the conflict began in 2014, thousands of people have been killed and over a...
